In [1]:
import pandas as pd
import numpy as np
import io
import glob
from collections import Counter
events_lines_path = "data/events_lines.json"
chunk_size = 500

# Общий счётчик типов событий
event_type_counter = Counter()

# Таблица "user_id × event_type" (сколько событий каждого типа у пользователя)
user_event_type_counts = None

with open(events_lines_path, "r", encoding="utf-8") as f:
    block = []
    for line in f:
        line = line.strip()
        if not line:
            continue

        block.append(line)

        if len(block) >= chunk_size:
            df_chunk = pd.read_json(io.StringIO("\n".join(block)), lines=True)
            df_chunk["event_time"] = pd.to_datetime(df_chunk["event_time"])

            # 1) Общая статистика по типам событий
            event_type_counter.update(df_chunk["event_type"])

            # 2) Статистика по пользователям и типам событий
            cnt = (
                df_chunk
                .groupby(["user_id", "event_type"])
                .size()
                .unstack(fill_value=0)
            )

            if user_event_type_counts is None:
                user_event_type_counts = cnt
            else:
                user_event_type_counts = user_event_type_counts.add(cnt, fill_value=0)

            block = []

    # Последний неполный блок
    if block:
        df_chunk = pd.read_json(io.StringIO("\n".join(block)), lines=True)
        df_chunk["event_time"] = pd.to_datetime(df_chunk["event_time"])

        event_type_counter.update(df_chunk["event_type"])

        cnt = (
            df_chunk
            .groupby(["user_id", "event_type"])
            .size()
            .unstack(fill_value=0)
        )

        if user_event_type_counts is None:
            user_event_type_counts = cnt
        else:
            user_event_type_counts = user_event_type_counts.add(cnt, fill_value=0)

In [2]:
event_type_stats = (
    pd.Series(event_type_counter, name="events_count")
    .sort_values(ascending=False)
)

event_type_stats.to_csv("events_type_stats.csv", header=True)

print("Сводная статистика по типам событий:")
print(event_type_stats)

Сводная статистика по типам событий:
click          416
purchase       413
scroll         405
add_to_cart    387
page_view      379
Name: events_count, dtype: int64


In [3]:
user_event_type_counts = (
    user_event_type_counts
    .fillna(0)
    .astype(int)
    .reset_index()
)

# Общее количество событий у пользователя
user_event_type_counts["events_total"] = (
    user_event_type_counts
    .drop(columns=["user_id"])
    .sum(axis=1)
)

user_event_type_counts.to_csv("user_event_type_stats.csv", index=False)

print("\nСтатистика событий по пользователям:")
print(user_event_type_counts.head())


Статистика событий по пользователям:
event_type  user_id  add_to_cart  click  page_view  purchase  scroll  \
0              1001            4      9          3        11       2   
1              1002            3      6          4         8       4   
2              1003            2      3          3         8       5   
3              1004            4      3          2         9       4   
4              1005            4      5          2         7       3   

event_type  events_total  
0                     29  
1                     25  
2                     21  
3                     22  
4                     21  


In [6]:
order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in order_files],
    ignore_index=True
)

print("\nЗаказы:")
print(df_orders.head())
print(df_orders.info())


Заказы:
   order_id  user_id      order_datetime  product_id     price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.15         1   
1         2     1079 2024-01-03 17:45:43          16   9172.60         1   
2         3     1041 2024-01-06 22:46:38          25   9035.26         2   
3         4     1083 2024-01-08 21:03:51           9  15234.33         3   
4         5     1068 2024-01-03 23:26:53          24   4134.38         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int64         
 1   user_id         15000 non-null  int64         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int64         
 4   price           15000 non-null  floa

In [7]:
df_orders_events = df_orders.merge(
    user_event_type_counts,
    on="user_id",
    how="left"
)

# Пользователи без событий получают 0
event_cols = [
    c for c in user_event_type_counts.columns
    if c not in ["user_id"]
]

df_orders_events[event_cols] = (
    df_orders_events[event_cols]
    .fillna(0)
    .astype(int)
)

# Фильтр: только пользователи с более чем 10 событиями
df_active = df_orders_events[
    df_orders_events["events_total"] > 10
].copy()

df_active.to_csv("orders_active_users.csv", index=False)

print("\nОбъединённая таблица по активным пользователям:")
print(df_active.head())
print("Размер:", df_active.shape)


Объединённая таблица по активным пользователям:
   order_id  user_id      order_datetime  product_id     price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.15         1   
1         2     1079 2024-01-03 17:45:43          16   9172.60         1   
2         3     1041 2024-01-06 22:46:38          25   9035.26         2   
3         4     1083 2024-01-08 21:03:51           9  15234.33         3   
4         5     1068 2024-01-03 23:26:53          24   4134.38         2   

     status  add_to_cart  click  page_view  purchase  scroll  events_total  
0  refunded            4      5          5         4       2            20  
1      paid            2      3          4         3       4            16  
2      paid            2      3          6         5       5            21  
3      paid            4      2          4         4       5            19  
4      paid            4      2          6         2       3            17  
Размер: (15000, 13)


In [8]:
active_users = sorted(df_active["user_id"].unique())

with open("active_users.txt", "w", encoding="utf-8") as f:
    for uid in active_users:
        f.write(f"{uid}\n")

print("\nКоличество активных пользователей:", len(active_users))
print("Первые 20 активных user_id:", active_users[:20])


Количество активных пользователей: 100
Первые 20 активных user_id: [np.int64(1001), np.int64(1002), np.int64(1003), np.int64(1004), np.int64(1005), np.int64(1006), np.int64(1007), np.int64(1008), np.int64(1009), np.int64(1010), np.int64(1011), np.int64(1012), np.int64(1013), np.int64(1014), np.int64(1015), np.int64(1016), np.int64(1017), np.int64(1018), np.int64(1019), np.int64(1020)]
